# Cleaning data tur konser

Notebook ini membaca file CSV lokal, membersihkan kolom teks/angka/tanggal, lalu menyimpan data bersih agar siap dipakai untuk analisis.

In [ ]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

raw_path = Path(r"C:\Users\Fahrul Rozi\Downloads\Dataset\my_file (1).csv")
clean_path = Path(r"C:\Users\Fahrul Rozi\Downloads\DataScience\my_file_clean.csv")

df_raw = pd.read_csv(raw_path)
df_raw.head()

In [ ]:
def clean_column_name(name):
    return (
        name.replace("\u00a0", " ")
        .strip()
        .lower()
        .replace("(in 2022 dollars)", "2022_usd")
        .replace(".", "")
        .replace(" ", "_")
    )


def parse_money(value):
    if pd.isna(value):
        return pd.NA

    digits = re.sub(r"[^0-9]", "", str(value))
    return int(digits) if digits else pd.NA


def parse_number(value):
    if pd.isna(value):
        return pd.NA

    match = re.search(r"\d+", str(value))
    return int(match.group()) if match else pd.NA


def clean_tour_title(value):
    if pd.isna(value):
        return value

    cleaned = re.sub(r"\[[^\]]+\]", "", str(value))
    cleaned = re.sub(r"[\u2020\u2021*]", "", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


df_clean = df_raw.copy()
df_clean.columns = [clean_column_name(col) for col in df_clean.columns]

df_clean = df_clean.rename(
    columns={
        "actual_gross": "actual_gross_usd",
        "adjusted_gross_2022_usd": "adjusted_gross_2022_usd",
        "average_gross": "average_gross_usd",
        "year(s)": "year_range",
        "ref": "references",
    }
)

money_columns = ["actual_gross_usd", "adjusted_gross_2022_usd", "average_gross_usd"]
for column in money_columns:
    df_clean[column] = df_clean[column].apply(parse_money).astype("Int64")

for column in ["peak", "all_time_peak"]:
    df_clean[column] = df_clean[column].apply(parse_number).astype("Int64")

year_parts = df_clean["year_range"].str.extract(r"(?P<start_year>\d{4})(?:\D+(?P<end_year>\d{4}))?")
df_clean["start_year"] = year_parts["start_year"].astype("Int64")
df_clean["end_year"] = year_parts["end_year"].fillna(year_parts["start_year"]).astype("Int64")
df_clean["duration_years"] = df_clean["end_year"] - df_clean["start_year"] + 1

df_clean["tour_title"] = df_clean["tour_title"].apply(clean_tour_title)
df_clean["artist"] = df_clean["artist"].str.strip()
df_clean["references"] = df_clean["references"].astype("string").str.strip()
df_clean["shows"] = df_clean["shows"].astype("Int64")

df_clean = df_clean.drop(columns=["peak", "all_time_peak", "references"])
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

df_clean.head()

In [ ]:
quality_check = pd.DataFrame(
    {
        "dtype": df_clean.dtypes.astype(str),
        "missing_values": df_clean.isna().sum(),
        "unique_values": df_clean.nunique(dropna=True),
    }
)

print(f"Jumlah baris duplikat: {df_clean.duplicated().sum()}")
quality_check

In [ ]:
try:
    df_clean.to_csv(clean_path, index=False)
    print(f"Data bersih tersimpan di: {clean_path}")
except PermissionError:
    backup_path = clean_path.with_name(f"{clean_path.stem}_baru{clean_path.suffix}")
    df_clean.to_csv(backup_path, index=False)
    print(f"File lama sedang terbuka, jadi data bersih tersimpan di: {backup_path}")

## Exploratory Data Analysis


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

dropped_clean_path = Path(r"C:\Users\Fahrul Rozi\Downloads\DataScience\my_file_clean_baru.csv")
eda_path = dropped_clean_path if dropped_clean_path.exists() else clean_path

df_eda = pd.read_csv(eda_path)

print(f"Data EDA dibaca dari: {eda_path}")
print(f"Jumlah baris: {df_eda.shape[0]}")
print(f"Jumlah kolom: {df_eda.shape[1]}")
df_eda.head()

## Feature Engineering Sederhana

Bagian ini membuat kolom tambahan untuk membantu analisis: gross per show, selisih gross setelah penyesuaian inflasi, dan kategori gross.

In [ ]:
df_eda["gross_per_show"] = df_eda["actual_gross_usd"] / df_eda["shows"]
df_eda["gross_difference"] = df_eda["adjusted_gross_2022_usd"] - df_eda["actual_gross_usd"]

df_eda["gross_category"] = pd.cut(
    df_eda["actual_gross_usd"],
    bins=[-float("inf"), 200_000_000, 400_000_000, float("inf")],
    labels=["Low", "Medium", "High"],
)

feature_path = Path(r"C:\Users\Fahrul Rozi\Downloads\DataScience\my_file_feature_engineered.csv")
df_eda.to_csv(feature_path, index=False)

print(f"Data dengan feature engineering tersimpan di: {feature_path}")
df_eda[["artist", "tour_title", "actual_gross_usd", "shows", "gross_per_show", "gross_difference", "gross_category"]].head()

In [ ]:
eda_quality = pd.DataFrame(
    {
        "dtype": df_eda.dtypes.astype(str),
        "missing_values": df_eda.isna().sum(),
        "unique_values": df_eda.nunique(dropna=True),
    }
)

print(f"Jumlah data duplikat: {df_eda.duplicated().sum()}")
eda_quality

In [ ]:
numeric_columns = [
    "rank",
    "actual_gross_usd",
    "adjusted_gross_2022_usd",
    "shows",
    "average_gross_usd",
    "start_year",
    "end_year",
    "duration_years",
    "gross_per_show",
    "gross_difference",
]

df_eda[numeric_columns].describe().T


In [ ]:
top_gross_tours = df_eda.sort_values("actual_gross_usd", ascending=False).head(10)
top_average_gross_tours = df_eda.sort_values("average_gross_usd", ascending=False).head(10)

display(top_gross_tours[["artist", "tour_title", "actual_gross_usd", "shows", "year_range"]])
display(top_average_gross_tours[["artist", "tour_title", "average_gross_usd", "shows", "year_range"]])

In [ ]:
artist_summary = (
    df_eda.groupby("artist", as_index=False)
    .agg(
        total_tours=("tour_title", "count"),
        total_actual_gross_usd=("actual_gross_usd", "sum"),
        avg_actual_gross_usd=("actual_gross_usd", "mean"),
        avg_gross_per_show_usd=("average_gross_usd", "mean"),
        total_shows=("shows", "sum"),
    )
    .sort_values("total_actual_gross_usd", ascending=False)
)

artist_summary

In [ ]:
year_summary = (
    df_eda.groupby("start_year", as_index=False)
    .agg(
        total_tours=("tour_title", "count"),
        total_actual_gross_usd=("actual_gross_usd", "sum"),
        avg_actual_gross_usd=("actual_gross_usd", "mean"),
    )
    .sort_values("start_year")
)

year_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(top_gross_tours["tour_title"], top_gross_tours["actual_gross_usd"])
plt.gca().invert_yaxis()
plt.title("Top 10 Tour Berdasarkan Actual Gross")
plt.xlabel("Actual Gross USD")
plt.ylabel("Tour Title")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(artist_summary["artist"], artist_summary["total_actual_gross_usd"])
plt.title("Total Actual Gross per Artist")
plt.xlabel("Artist")
plt.ylabel("Total Actual Gross USD")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_eda["shows"], df_eda["actual_gross_usd"])
plt.title("Hubungan Jumlah Shows dan Actual Gross")
plt.xlabel("Shows")
plt.ylabel("Actual Gross USD")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(year_summary["start_year"], year_summary["total_actual_gross_usd"], marker="o")
plt.title("Total Actual Gross Berdasarkan Tahun Mulai Tour")
plt.xlabel("Start Year")
plt.ylabel("Total Actual Gross USD")
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = df_eda[numeric_columns].corr()

plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, cmap="viridis")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

correlation_matrix

## Data Visualization: Memahami Cerita Data

Visualisasi di bawah fokus pada pertanyaan utama: tur mana yang menghasilkan uang paling besar, artist mana yang paling dominan, dan apakah jumlah show berhubungan dengan total gross.

In [ ]:
from matplotlib.ticker import FuncFormatter

def usd_millions(value, position=None):
    return f"${value / 1_000_000:.0f}M"


def usd_billions(value):
    return f"${value / 1_000_000_000:.2f}B"


plot_df = df_eda.copy()
plot_df["actual_gross_million"] = plot_df["actual_gross_usd"] / 1_000_000
plot_df["average_gross_million"] = plot_df["average_gross_usd"] / 1_000_000
plot_df["adjustment_gap_usd"] = plot_df["adjusted_gross_2022_usd"] - plot_df["actual_gross_usd"]
plot_df["adjustment_gap_million"] = plot_df["adjustment_gap_usd"] / 1_000_000

top_tour = plot_df.loc[plot_df["actual_gross_usd"].idxmax()]
best_per_show = plot_df.loc[plot_df["average_gross_usd"].idxmax()]
longest_tour = plot_df.loc[plot_df["shows"].idxmax()]
top_artist = artist_summary.iloc[0]

print("Insight cepat:")
print(f"1. Tour terbesar: {top_tour['tour_title']} oleh {top_tour['artist']} ({usd_billions(top_tour['actual_gross_usd'])}).")
print(f"2. Artist dengan total gross terbesar: {top_artist['artist']} ({usd_billions(top_artist['total_actual_gross_usd'])}).")
print(f"3. Gross per show tertinggi: {best_per_show['tour_title']} ({usd_millions(best_per_show['average_gross_usd'])} per show).")
print(f"4. Tour dengan show terbanyak: {longest_tour['tour_title']} ({longest_tour['shows']} shows).")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Dashboard Ringkas Tur Konser", fontsize=16, fontweight="bold")

top_10 = plot_df.sort_values("actual_gross_usd", ascending=True).tail(10)
axes[0, 0].barh(top_10["tour_title"], top_10["actual_gross_usd"], color="#2F6F9F")
axes[0, 0].set_title("Top 10 Tour Berdasarkan Total Gross")
axes[0, 0].set_xlabel("Actual Gross")
axes[0, 0].xaxis.set_major_formatter(FuncFormatter(usd_millions))

artist_top = artist_summary.sort_values("total_actual_gross_usd", ascending=True)
axes[0, 1].barh(artist_top["artist"], artist_top["total_actual_gross_usd"], color="#7A4EAB")
axes[0, 1].set_title("Total Gross per Artist")
axes[0, 1].set_xlabel("Total Actual Gross")
axes[0, 1].xaxis.set_major_formatter(FuncFormatter(usd_millions))

axes[1, 0].scatter(
    plot_df["shows"],
    plot_df["actual_gross_usd"],
    s=plot_df["average_gross_million"] * 45,
    alpha=0.7,
    color="#D86C4A",
    edgecolor="black",
    linewidth=0.5,
)
axes[1, 0].set_title("Jumlah Shows vs Total Gross")
axes[1, 0].set_xlabel("Jumlah Shows")
axes[1, 0].set_ylabel("Actual Gross")
axes[1, 0].yaxis.set_major_formatter(FuncFormatter(usd_millions))

for _, row in plot_df.nlargest(4, "actual_gross_usd").iterrows():
    axes[1, 0].annotate(row["artist"], (row["shows"], row["actual_gross_usd"]), xytext=(5, 5), textcoords="offset points", fontsize=9)

axes[1, 1].plot(year_summary["start_year"], year_summary["total_actual_gross_usd"], marker="o", color="#2A9D8F")
axes[1, 1].set_title("Total Gross Berdasarkan Tahun Mulai Tour")
axes[1, 1].set_xlabel("Start Year")
axes[1, 1].set_ylabel("Total Actual Gross")
axes[1, 1].yaxis.set_major_formatter(FuncFormatter(usd_millions))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

efficient_tours = plot_df.sort_values("average_gross_usd", ascending=True).tail(10)
axes[0].barh(efficient_tours["tour_title"], efficient_tours["average_gross_usd"], color="#3B8C66")
axes[0].set_title("Top 10 Gross Rata-rata per Show")
axes[0].set_xlabel("Average Gross per Show")
axes[0].xaxis.set_major_formatter(FuncFormatter(usd_millions))

adjustment_top = plot_df.sort_values("adjustment_gap_usd", ascending=True).tail(10)
axes[1].barh(adjustment_top["tour_title"], adjustment_top["adjustment_gap_usd"], color="#C58B2B")
axes[1].set_title("Kenaikan Gross Setelah Disesuaikan ke Dollar 2022")
axes[1].set_xlabel("Adjusted Gross - Actual Gross")
axes[1].xaxis.set_major_formatter(FuncFormatter(usd_millions))

plt.tight_layout()
plt.show()

In [ ]:
corr_shows_gross = plot_df["shows"].corr(plot_df["actual_gross_usd"])
corr_avg_total = plot_df["average_gross_usd"].corr(plot_df["actual_gross_usd"])

print("Cara membaca hasil visualisasi:")
print(f"- Korelasi jumlah shows vs total gross: {corr_shows_gross:.2f}. Artinya jumlah show tidak otomatis menjamin gross paling besar.")
print(f"- Korelasi average gross per show vs total gross: {corr_avg_total:.2f}. Ini menunjukkan efisiensi per show sangat berpengaruh pada total gross.")
print("- Tour modern seperti The Eras Tour dan Renaissance World Tour sangat kuat karena gross per show-nya tinggi, meskipun jumlah show tidak sebanyak tour lama.")
print("- Tour lama seperti Living Proof punya show sangat banyak, tetapi average gross per show-nya jauh lebih rendah dibanding tour modern.")

## Insight Untuk Setiap Visualisasi

### 1. Top 10 Tour Berdasarkan Total Gross
- Temuan:
  The Eras Tour menjadi tur dengan actual gross tertinggi, diikuti Renaissance World Tour dan Sticky & Sweet Tour.
- Makna:
  Pendapatan terbesar terkonsentrasi pada beberapa tur papan atas; tidak semua tur dalam daftar memiliki skala pendapatan yang sama.
- Kemungkinan penyebab:
  Demand tinggi, harga tiket premium, kapasitas venue besar, dan popularitas artist saat tur berlangsung.

### 2. Total Gross per Artist
- Temuan:
  Taylor Swift menjadi artist dengan total actual gross terbesar di dataset.
- Makna:
  Dominasi Taylor Swift bukan hanya dari satu tur, tetapi dari beberapa tur besar yang konsisten masuk daftar.
- Kemungkinan penyebab:
  Basis penggemar global, strategi tur stadion, katalog album yang kuat, dan permintaan tiket yang sangat tinggi.

### 3. Jumlah Shows vs Total Gross
- Temuan:
  Jumlah show tidak selalu sejalan dengan total gross. Korelasi shows terhadap actual gross bernilai negatif lemah.
- Makna:
  Banyak show bukan jaminan menjadi tur paling menguntungkan. Gross per show lebih menentukan.
- Kemungkinan penyebab:
  Perbedaan harga tiket, kapasitas venue, era tur, negara/kota yang dikunjungi, dan daya beli penonton.

### 4. Total Gross Berdasarkan Tahun Mulai Tour
- Temuan:
  Tahun-tahun terbaru, terutama 2023, terlihat sangat kuat karena ada tur besar seperti The Eras Tour dan Renaissance World Tour.
- Makna:
  Pasar konser modern menghasilkan gross yang jauh lebih tinggi dibanding banyak tur lama.
- Kemungkinan penyebab:
  Inflasi harga tiket, venue lebih besar, sistem penjualan tiket modern, dan meningkatnya demand konser pascapandemi.

### 5. Top 10 Gross Rata-rata per Show
- Temuan:
  The Eras Tour memiliki gross per show tertinggi, disusul Renaissance World Tour.
- Makna:
  Efisiensi pendapatan per konser sangat tinggi pada tur modern tertentu.
- Kemungkinan penyebab:
  Tiket premium, stadion besar, banyak show sold out, dan tingginya willingness to pay dari penonton.

### 6. Kenaikan Gross Setelah Disesuaikan ke Dollar 2022
- Temuan:
  Beberapa tur lama mengalami kenaikan besar setelah gross disesuaikan ke nilai dollar 2022.
- Makna:
  Perbandingan actual gross saja kurang adil untuk tur lintas era; adjusted gross membantu membaca nilai historis.
- Kemungkinan penyebab:
  Inflasi, perubahan harga tiket, dan perbedaan nilai uang antara tahun tur lama dan 2022.

### 7. Correlation Matrix
- Temuan:
  Average gross per show memiliki hubungan paling kuat dengan actual gross, sedangkan jumlah shows tidak menunjukkan hubungan positif kuat.
- Makna:
  Kualitas pendapatan per show lebih penting daripada sekadar kuantitas show.
- Kemungkinan penyebab:
  Tur dengan sedikit show tetapi harga tiket dan kapasitas venue tinggi dapat mengalahkan tur dengan show lebih banyak.

## Kesimpulan Akhir Project

In [ ]:
highest_gross_tour = df_eda.loc[df_eda["actual_gross_usd"].idxmax()]
dominant_artist = artist_summary.iloc[0]
best_average_tour = df_eda.loc[df_eda["gross_per_show"].idxmax()]
shows_gross_corr = df_eda["shows"].corr(df_eda["actual_gross_usd"])

print("Kesimpulan akhir project:")
print(f"1. Tour dengan gross tertinggi adalah {highest_gross_tour['tour_title']} oleh {highest_gross_tour['artist']} dengan actual gross ${highest_gross_tour['actual_gross_usd']:,.0f}.")
print(f"2. Artist paling dominan adalah {dominant_artist['artist']} dengan total actual gross ${dominant_artist['total_actual_gross_usd']:,.0f}.")
print(f"3. Jumlah shows tidak selalu menentukan gross tertinggi. Korelasi shows dan actual gross adalah {shows_gross_corr:.2f}.")
print(f"4. Tour dengan average gross terbaik adalah {best_average_tour['tour_title']} oleh {best_average_tour['artist']} dengan gross per show ${best_average_tour['gross_per_show']:,.0f}.")
print("5. Saran analisis lanjutan: tambahkan data harga tiket rata-rata, kapasitas venue, negara/kota tur, jumlah penonton, dan periode penjualan tiket agar faktor penyebab gross tinggi bisa dianalisis lebih dalam.")